In [1]:
!pip install openfermion==1.7.1 pyscf==2.12.1 openfermionpyscf==0.5 torch==2.10.0 numpy==2.0.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.6/51.6 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 105.7 MB/s eta 0:00:00


The following implementation is unfair implementation of QuatumDARTS, but it is much better than the one with fixes. For completeness, I include this version as well

```
=== STRICTLY FAIR BENCHMARK: NOISE RESILIENCE (DEPTH 4) ===
Loaded LiH (2.5 Å) Hamiltonian. 4-qubit ground state: -7.773544 Ha

Method     | Shots    | Energy (Ha)  | Error (mHa)  | Gates
-----------------------------------------------------------------
DLP        | Exact    |    -7.77223 |      1.317   | 11    
Q-DARTS    | Exact    |    -7.76959 |      3.958   | 15    
DLP        | 10000    |    -7.77196 |      1.586   | 11    
Q-DARTS    | 10000    |    -7.76628 |      7.264   | 19    
DLP        | 1000     |    -7.77187 |      1.669   | 9     
Q-DARTS    | 1000     |    -7.69399 |     79.551   | 18    
DLP        | 100      |    -7.77193 |      1.611   | 14    
Q-DARTS    | 100      |    -7.75996 |     13.587   | 30    
DLP        | 50       |    -7.77129 |      2.250   | 13    
Q-DARTS    | 50       |    -7.74467 |     28.871   | 23  
```
# These are fixes

```
    with torch.no_grad():
        U, switches = ansatz(temperature=0.05)
```
For DLP: `torch.sigmoid(self.logit)` is mathematically deterministic. It evaluates the exact learned state of the circuit.
For Q-DARTS: `F.gumbel_softmax(self.alphas, tau=temperature, hard=False)` ALWAYS samples from a Gumbel distribution, even inside a `torch.no_grad()` block.
The Impact: You are evaluating Q-DARTS on a "wobbly", randomized circuit. Because Gumbel noise is injected during the final evaluation, Q-DARTS's energy will be artificially higher (worse) and its gate count will randomly fluctuate.

The Fix: In standard DARTS literature, evaluation must be deterministic. You should use standard Softmax or argmax during inference for Q-DARTS to remove the noise.
python

### Inside LinearRelaxationGate.forward()
```
if self.method == 'Q-DARTS':
    if self.training: # You'll need to pass training state or check it
        w = F.gumbel_softmax(self.alphas, tau=temperature, hard=False)
    else:
        # Deterministic evaluation without Gumbel noise
        w = F.softmax(self.alphas / temperature, dim=0)
    return w[0] * I + w[1] * U, w[1]
```

2. The Double Penalty of Noise (Moderate)
During training, Q-DARTS uses F.gumbel_softmax, which inherently injects structural stochasticity into the forward pass. When you add simulated measurement shot noise directly to the gradients (`p.grad += torch.randn_like(...)`), Q-DARTS is being double-penalized. It suffers from the variance of Gumbel sampling plus the variance of shot noise. DLP, using a deterministic forward pass, only suffers from the shot noise.
While this is technically how the two algorithms naturally react to shot noise, reviewers might point out that Gumbel-Softmax was originally designed for noiseless exact gradients (like in classical Neural Architecture Search).
3. Continuous Unitarity vs. Discrete Sampling (Moderate)
In the Q-DARTS paper, the algorithm uses Gumbel-Softmax to select exactly one gate (using argmax in the forward pass but softmax in the backward pass, essentially the hard=True argument in PyTorch). Your implementation uses hard=False, meaning Q-DARTS is acting as a continuous linear relaxation rather than a discrete sampler. If you want to benchmark against the true Q-DARTS, you should set hard=True so it behaves as intended by its authors: guarantees strict unitarity during the forward pass.

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import hashlib


from openfermion.chem import MolecularData
from openfermionpyscf import run_pyscf
from openfermion.transforms import get_fermion_operator, jordan_wigner
from openfermion.linalg import get_sparse_operator

# ============================================================================
# Core Components (Quantum Gates & Continuous Relaxation)
# ============================================================================

def kron(A, B):
    return torch.einsum("ab,cd->acbd", A, B).reshape(
        A.size(0) * B.size(0), A.size(1) * B.size(1)
    )

def tensor_product(ops):
    result = ops[0]
    for op in ops[1:]:
        result = kron(result, op)
    return result

class BaseQuantumGate(nn.Module):
    def __init__(self, n_qubits, target, gate_type, control=None):
        super().__init__()
        self.n_qubits = n_qubits
        self.target = target
        self.gate_type = gate_type
        self.control = control

        if gate_type in ['Ry', 'Rz']:
            self.theta = nn.Parameter(
                torch.tensor([np.random.normal(0, 0.1)], dtype=torch.float32)
            )
        else:
            self.theta = None

    def get_unitary(self):
        if self.gate_type == 'CNOT':
            d = 2 ** self.n_qubits
            U = torch.eye(d, dtype=torch.complex64)
            for i in range(d):
                if (i >> (self.n_qubits - 1 - self.control)) & 1:
                    j = i ^ (1 << (self.n_qubits - 1 - self.target))
                    U[i, i] = 0
                    U[i, j] = 1
            return U
        else:
            t = self.theta.squeeze()
            if self.gate_type == 'Ry':
                c, s = torch.cos(t / 2), torch.sin(t / 2)
                U_local = torch.stack([
                    torch.stack([c, -s]),
                    torch.stack([s, c])
                ]).to(torch.complex64)
            elif self.gate_type == 'Rz':
                U_local = torch.stack([
                    torch.stack([torch.exp(-1j * t / 2), torch.tensor(0j)]),
                    torch.stack([torch.tensor(0j), torch.exp(1j * t / 2)])
                ])
            ops = [torch.eye(2, dtype=torch.complex64)] * self.n_qubits
            ops[self.target] = U_local
            return tensor_product(ops)

class LinearRelaxationGate(nn.Module):
    def __init__(self, base_gate, method='DLP'):
        super().__init__()
        self.base_gate = base_gate
        self.method = method
        if method == 'DLP':
            self.logit = nn.Parameter(torch.tensor(0.0))
        elif method == 'Q-DARTS':
            self.alphas = nn.Parameter(torch.tensor([0.0, 0.0]))

    def forward(self, temperature=1.0):
        U = self.base_gate.get_unitary()
        I = torch.eye(U.size(0), dtype=torch.complex64)
        if self.method == 'DLP':
            s = torch.sigmoid(self.logit)
            return (1 - s) * I + s * U, s
        elif self.method == 'Q-DARTS':
            w = F.gumbel_softmax(self.alphas, tau=temperature, hard=False)
            return w[0] * I + w[1] * U, w[1]


# ============================================================================
# Deep Ansatz Builder
# ============================================================================

class DLPAnsatz(nn.Module):
    def __init__(self, n_qubits, method='DLP'):
        super().__init__()
        self.n_qubits = n_qubits
        self.method = method
        self.gates = nn.ModuleList()

    def add_gate(self, gate_type, target, control=None):
        base_gate = BaseQuantumGate(self.n_qubits, target, gate_type, control)
        self.gates.append(LinearRelaxationGate(base_gate, self.method))

    def build_ansatz(self, hf_bitstring, depth=4):
        """Hardware-Efficient Ansatz initialized with HF state."""
        # 1. Hartree-Fock Initialization
        for q, bit in enumerate(hf_bitstring):
            if bit == '1':
                self.add_gate('Ry', q)
                idx = len(self.gates) - 1
                # Initialize to pi, but DO NOT FREEZE them
                with torch.no_grad():
                    self.gates[idx].base_gate.theta.fill_(np.pi)
                    if self.method == 'DLP':
                        self.gates[idx].logit.fill_(4.0) # Start 'ON'
                    elif self.method == 'Q-DARTS':
                        self.gates[idx].alphas.copy_(torch.tensor([-4.0, 4.0]))

        # 2. Deep HEA Layers
        for d in range(depth):
            for q in range(self.n_qubits):
                self.add_gate('Ry', q)
                self.add_gate('Rz', q)
            for q in range(self.n_qubits - 1):
                self.add_gate('CNOT', q + 1, control=q)

    def forward(self, temperature=1.0):
        U = torch.eye(2 ** self.n_qubits, dtype=torch.complex64)
        switches = []
        for gate in self.gates:
            U_gate, s = gate(temperature)
            U = torch.matmul(U_gate, U)
            switches.append(s)
        return U, torch.stack(switches) if switches else torch.tensor([1.0])


# ============================================================================
# Fair Training Loop (No Oracles, Full Noise Injection, Annealing Curriculum)
# ============================================================================

def get_observable_variance(psi, H):
    exp_H = torch.real(torch.vdot(psi, torch.matmul(H, psi)))
    exp_H2 = torch.real(torch.vdot(psi, torch.matmul(torch.matmul(H, H), psi)))
    return max((exp_H2 - exp_H ** 2).item(), 0.0)

def train_fair(ansatz, H_hard, n_shots, epochs=1500, lr=0.02):

    # Differentiate param groups, but everything remains fully trainable
    angle_params = [p for n, p in ansatz.named_parameters() if 'theta' in n]
    struct_params = [p for n, p in ansatz.named_parameters() if 'logit' in n or 'alphas' in n]

    optimizer = optim.Adam([
        {'params': angle_params, 'lr': lr},
        {'params': struct_params, 'lr': lr * 0.5},
    ], betas=(0.9, 0.999))

    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=lr * 0.01)

    # Create H_easy (Diagonal elements only - breaks entanglement traps)
    H_easy = torch.diag(torch.diagonal(H_hard))

    anneal_epochs = int(epochs * 0.5)

    for epoch in range(epochs):
        # 1. Temperature Schedule
        temperature = max(0.05, 2.0 * (1 - epoch / epochs))

        # 2. Curriculum Schedules
        # Phase 1: Morph Hamiltonian (Epochs 0 -> 50%)
        t = min(1.0, epoch / anneal_epochs)
        H_current = (1 - t) * H_easy + t * H_hard

        # Phase 2: Simplicity Pruning (Epochs 50% -> 100%)
        if epoch < anneal_epochs:
            w_simp = 0.0
        else:
            progress = (epoch - anneal_epochs) / (epochs - anneal_epochs)
            w_simp_final = 0.005
            w_simp = w_simp_final / (1 + np.exp(-10 * (progress - 0.5))) # Sigmoid ramp

        optimizer.zero_grad()

        U, switches = ansatz(temperature)
        psi = U[:, 0]
        psi = psi / torch.norm(psi)

        E_exact = torch.real(torch.vdot(psi, torch.matmul(H_current, psi)))
        loss = E_exact + w_simp * switches.sum()
        loss.backward()

        # 3. FAIR NOISE INJECTION (Applied to ALL Gradients)
        if n_shots is not None and n_shots != float('inf'):
            variance = get_observable_variance(psi, H_current)
            noise_std = np.sqrt(variance / n_shots) if n_shots > 0 else 0.0

            with torch.no_grad():
                for p in ansatz.parameters():  # Impacts angles AND architectural logits/alphas!
                    if p.grad is not None:
                        p.grad += torch.randn_like(p.grad) * (noise_std / np.sqrt(2))

        # Gradient clipping to prevent noise explosions
        torch.nn.utils.clip_grad_norm_(ansatz.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

    # FINAL EVALUATION: No oracle reverting. We evaluate the exact final state
    # of the model against the true Hamiltonian.
    with torch.no_grad():
        U, switches = ansatz(temperature=0.05)
        psi = U[:, 0]
        psi = psi / torch.norm(psi)
        E_final = torch.real(torch.vdot(psi, torch.matmul(H_hard, psi))).item()
        gates_final = (switches > 0.5).sum().item()

    return E_final, gates_final


# ============================================================================
# Main Execution
# ============================================================================

def run_experiment():
    print("=== STRICTLY FAIR BENCHMARK: NOISE RESILIENCE (DEPTH 4) ===")

    if True:
        mol = MolecularData([['Li', [0, 0, 0]], ['H', [0, 0, 2.5]]], 'sto-3g', 1, 0)
        mol = run_pyscf(mol, run_scf=True, run_fci=True)
        ham = jordan_wigner(get_fermion_operator(
            mol.get_molecular_hamiltonian(occupied_indices=[0], active_indices=[1, 2])
        ))
        H_hard = torch.tensor(get_sparse_operator(ham).toarray(), dtype=torch.complex64)
        E_fci = torch.linalg.eigvalsh(H_hard).real[0].item()
        hf_bitstring = '1100'
        print(f"Loaded LiH (2.5 Å) Hamiltonian. 4-qubit ground state: {E_fci:.6f} Ha")

    noise_levels = [None, 10000, 1000, 100, 50]
    epochs = 1500

    print(f"\n{'Method':<10} | {'Shots':<8} | {'Energy (Ha)':<12} | {'Error (mHa)':<12} | {'Gates':<6}")
    print("-" * 65)

    for shots in noise_levels:
        shot_label = str(shots) if shots else "Exact"

        for method in ['DLP', 'Q-DARTS']:
            seed_str = f"fair_bench_{shots}_{method}"
            seed = int(hashlib.md5(seed_str.encode()).hexdigest(), 16) % (2**32)

            torch.manual_seed(seed)
            np.random.seed(seed)

            ansatz = DLPAnsatz(n_qubits=4, method=method)
            ansatz.build_ansatz(hf_bitstring, depth=4)

            # Train and return the true final state (no oracle saving)
            E, gates = train_fair(ansatz, H_hard, shots, epochs=epochs)

            error_mHa = (E - E_fci) * 1000
            print(f"{method:<10} | {shot_label:<8} | {E:>11.5f} | {error_mHa:>10.3f}   | {gates:<6}")

if __name__ == "__main__":
    run_experiment()

=== STRICTLY FAIR BENCHMARK: NOISE RESILIENCE (DEPTH 4) ===
Loaded LiH (2.5 Å) Hamiltonian. 4-qubit ground state: -7.773544 Ha

Method     | Shots    | Energy (Ha)  | Error (mHa)  | Gates 
-----------------------------------------------------------------
DLP        | Exact    |    -7.77223 |      1.317   | 11    
Q-DARTS    | Exact    |    -7.76959 |      3.958   | 15    
DLP        | 10000    |    -7.77196 |      1.586   | 11    
Q-DARTS    | 10000    |    -7.76628 |      7.264   | 19    
DLP        | 1000     |    -7.77187 |      1.669   | 9     
Q-DARTS    | 1000     |    -7.69399 |     79.551   | 18    
DLP        | 100      |    -7.77193 |      1.611   | 14    
Q-DARTS    | 100      |    -7.75996 |     13.587   | 30    
DLP        | 50       |    -7.77129 |      2.250   | 13    
Q-DARTS    | 50       |    -7.74467 |     28.871   | 23    


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import hashlib
import random

from openfermion.chem import MolecularData
from openfermionpyscf import run_pyscf
from openfermion.transforms import get_fermion_operator, jordan_wigner
from openfermion.linalg import get_sparse_operator
# ============================================================================
# Core Components (Quantum Gates & Continuous Relaxation)
# ============================================================================

def set_seed(seed):
    """Ensure fully deterministic runs."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def kron(A, B):
    return torch.einsum("ab,cd->acbd", A, B).reshape(
        A.size(0) * B.size(0), A.size(1) * B.size(1)
    )

def tensor_product(ops):
    result = ops[0]
    for op in ops[1:]:
        result = kron(result, op)
    return result

class BaseQuantumGate(nn.Module):
    def __init__(self, n_qubits, target, gate_type, control=None):
        super().__init__()
        self.n_qubits = n_qubits
        self.target = target
        self.gate_type = gate_type
        self.control = control

        if gate_type in ['Ry', 'Rz']:
            self.theta = nn.Parameter(
                torch.tensor([np.random.normal(0, 0.1)], dtype=torch.float32)
            )
        else:
            self.theta = None

    def get_unitary(self):
        if self.gate_type == 'CNOT':
            d = 2 ** self.n_qubits
            U = torch.eye(d, dtype=torch.complex64)
            for i in range(d):
                if (i >> (self.n_qubits - 1 - self.control)) & 1:
                    j = i ^ (1 << (self.n_qubits - 1 - self.target))
                    U[i, i] = 0
                    U[i, j] = 1
            return U
        else:
            t = self.theta.squeeze()
            if self.gate_type == 'Ry':
                c, s = torch.cos(t / 2), torch.sin(t / 2)
                U_local = torch.stack([
                    torch.stack([c, -s]),
                    torch.stack([s, c])
                ]).to(torch.complex64)
            elif self.gate_type == 'Rz':
                U_local = torch.stack([
                    torch.stack([torch.exp(-1j * t / 2), torch.tensor(0j)]),
                    torch.stack([torch.tensor(0j), torch.exp(1j * t / 2)])
                ])
            ops = [torch.eye(2, dtype=torch.complex64)] * self.n_qubits
            ops[self.target] = U_local
            return tensor_product(ops)

class LinearRelaxationGate(nn.Module):
    def __init__(self, base_gate, method='DLP'):
        super().__init__()
        self.base_gate = base_gate
        self.method = method
        if method == 'DLP':
            self.logit = nn.Parameter(torch.tensor(0.0))
        elif method == 'Q-DARTS':
            self.alphas = nn.Parameter(torch.tensor([0.0, 0.0]))

    def forward(self, temperature=1.0):
        U = self.base_gate.get_unitary()
        I = torch.eye(U.size(0), dtype=torch.complex64)

        if self.method == 'DLP':
            # DLP uses standard smooth sigmoid
            s = torch.sigmoid(self.logit)
            return (1 - s) * I + s * U, s

        elif self.method == 'Q-DARTS':
            if self.training:
                # FIX: hard=True ensures strict unitarity in forward pass
                # using the Straight-Through Estimator, matching wu23v.pdf Eq 3.
                w = F.gumbel_softmax(self.alphas, tau=temperature, hard=True)
            else:
                # FIX: Deterministic evaluation without Gumbel noise
                w = torch.zeros_like(self.alphas)
                w[self.alphas.argmax()] = 1.0

            return w[0] * I + w[1] * U, w[1]


# ============================================================================
# Deep Ansatz Builder
# ============================================================================

class DLPAnsatz(nn.Module):
    def __init__(self, n_qubits, method='DLP'):
        super().__init__()
        self.n_qubits = n_qubits
        self.method = method
        self.gates = nn.ModuleList()

    def add_gate(self, gate_type, target, control=None):
        base_gate = BaseQuantumGate(self.n_qubits, target, gate_type, control)
        self.gates.append(LinearRelaxationGate(base_gate, self.method))

    def build_ansatz(self, hf_bitstring, depth=4):
        """Hardware-Efficient Ansatz initialized with HF state."""
        # 1. Hartree-Fock Initialization
        for q, bit in enumerate(hf_bitstring):
            if bit == '1':
                self.add_gate('Ry', q)
                idx = len(self.gates) - 1
                with torch.no_grad():
                    self.gates[idx].base_gate.theta.fill_(np.pi)
                    if self.method == 'DLP':
                        self.gates[idx].logit.fill_(4.0)
                    elif self.method == 'Q-DARTS':
                        self.gates[idx].alphas.copy_(torch.tensor([-4.0, 4.0]))

        # 2. Deep HEA Layers
        for d in range(depth):
            for q in range(self.n_qubits):
                self.add_gate('Ry', q)
                self.add_gate('Rz', q)
            for q in range(self.n_qubits - 1):
                self.add_gate('CNOT', q + 1, control=q)

    def forward(self, temperature=1.0):
        U = torch.eye(2 ** self.n_qubits, dtype=torch.complex64)
        switches = []
        for gate in self.gates:
            U_gate, s = gate(temperature)
            U = torch.matmul(U_gate, U)
            switches.append(s)
        return U, torch.stack(switches) if switches else torch.tensor([1.0])


# ============================================================================
# Fair Training Loop
# ============================================================================

def get_observable_variance(psi, H):
    exp_H = torch.real(torch.vdot(psi, torch.matmul(H, psi)))
    exp_H2 = torch.real(torch.vdot(psi, torch.matmul(torch.matmul(H, H), psi)))
    return max((exp_H2 - exp_H ** 2).item(), 0.0)

def train_fair(ansatz, H_hard, n_shots, epochs=1500, lr=0.02):

    # Put model in training mode (activates Gumbel-Softmax noise for Q-DARTS)
    ansatz.train()

    angle_params = [p for n, p in ansatz.named_parameters() if 'theta' in n]
    struct_params = [p for n, p in ansatz.named_parameters() if 'logit' in n or 'alphas' in n]

    optimizer = optim.Adam([
        {'params': angle_params, 'lr': lr},
        {'params': struct_params, 'lr': lr * 0.5},
    ], betas=(0.9, 0.999))

    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=lr * 0.01)

    H_easy = torch.diag(torch.diagonal(H_hard))
    anneal_epochs = int(epochs * 0.5)

    for epoch in range(epochs):
        temperature = max(0.05, 2.0 * (1 - epoch / epochs))

        t = min(1.0, epoch / anneal_epochs)
        H_current = (1 - t) * H_easy + t * H_hard

        if epoch < anneal_epochs:
            w_simp = 0.0
        else:
            progress = (epoch - anneal_epochs) / (epochs - anneal_epochs)
            w_simp_final = 0.005
            w_simp = w_simp_final / (1 + np.exp(-10 * (progress - 0.5)))

        optimizer.zero_grad()

        U, switches = ansatz(temperature)
        psi = U[:, 0]
        psi = psi / torch.norm(psi)

        E_exact = torch.real(torch.vdot(psi, torch.matmul(H_current, psi)))
        loss = E_exact + w_simp * switches.sum()
        loss.backward()

        # FAIR NOISE INJECTION
        if n_shots is not None and n_shots != float('inf'):
            variance = get_observable_variance(psi, H_current)
            noise_std = np.sqrt(variance / n_shots) if n_shots > 0 else 0.0

            with torch.no_grad():
                for p in ansatz.parameters():
                    if p.grad is not None:
                        p.grad += torch.randn_like(p.grad) * (noise_std / np.sqrt(2))

        torch.nn.utils.clip_grad_norm_(ansatz.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

    # FINAL EVALUATION: Deterministic Check
    # Disable Gumbel sampling so Q-DARTS locks in its highest-probability structure
    ansatz.eval()
    with torch.no_grad():
        U, switches = ansatz(temperature=0.05)
        psi = U[:, 0]
        psi = psi / torch.norm(psi)
        E_final = torch.real(torch.vdot(psi, torch.matmul(H_hard, psi))).item()
        gates_final = (switches > 0.5).sum().item()

    return E_final, gates_final


# ============================================================================
# Main Execution
# ============================================================================

def run_experiment():
    print("=== BENCHMARK: NOISE RESILIENCE (DEPTH 4) ===")

    if True:
        mol = MolecularData([['Li', [0, 0, 0]], ['H', [0, 0, 2.5]]], 'sto-3g', 1, 0)
        mol = run_pyscf(mol, run_scf=True, run_fci=True)
        ham = jordan_wigner(get_fermion_operator(
            mol.get_molecular_hamiltonian(occupied_indices=[0], active_indices=[1, 2])
        ))
        H_hard = torch.tensor(get_sparse_operator(ham).toarray(), dtype=torch.complex64)
        E_fci = torch.linalg.eigvalsh(H_hard).real[0].item()
        hf_bitstring = '1100'
        print(f"Loaded LiH (2.5 Å) Hamiltonian. 4-qubit ground state: {E_fci:.6f} Ha")

    noise_levels = [None, 10000, 1000, 100, 50]
    epochs = 1500

    print(f"\n{'Method':<10} | {'Shots':<8} | {'Energy (Ha)':<12} | {'Error (mHa)':<12} | {'Gates':<6}")
    print("-" * 65)

    for shots in noise_levels:
        shot_label = str(shots) if shots else "Exact"

        for method in ['DLP', 'Q-DARTS']:
            # Create a deterministic seed for this specific combination
            seed_str = f"fair_bench_{shots}_{method}"
            seed = int(hashlib.md5(seed_str.encode()).hexdigest(), 16) % (2**32)

            # Apply strict seeding
            set_seed(seed)

            ansatz = DLPAnsatz(n_qubits=4, method=method)
            ansatz.build_ansatz(hf_bitstring, depth=4)

            E, gates = train_fair(ansatz, H_hard, shots, epochs=epochs)

            error_mHa = (E - E_fci) * 1000
            print(f"{method:<10} | {shot_label:<8} | {E:>11.5f} | {error_mHa:>10.3f}   | {gates:<6}")

if __name__ == "__main__":
    run_experiment()

=== STRICTLY FAIR BENCHMARK: NOISE RESILIENCE (DEPTH 4) ===
Loaded LiH (2.5 Å) Hamiltonian. 4-qubit ground state: -7.773544 Ha

Method     | Shots    | Energy (Ha)  | Error (mHa)  | Gates 
-----------------------------------------------------------------
DLP        | Exact    |    -7.77223 |      1.317   | 11    
Q-DARTS    | Exact    |    -7.70690 |     66.648   | 11    
DLP        | 10000    |    -7.77196 |      1.586   | 11    
Q-DARTS    | 10000    |    -7.67930 |     94.242   | 17    
DLP        | 1000     |    -7.77187 |      1.669   | 9     
Q-DARTS    | 1000     |    -7.70705 |     66.492   | 9     
DLP        | 100      |    -7.77193 |      1.611   | 14    
Q-DARTS    | 100      |    -7.67672 |     96.819   | 22    
DLP        | 50       |    -7.77129 |      2.250   | 13    
Q-DARTS    | 50       |    -7.60494 |    168.599   | 25    
